<div align="center">

# Prepare weighted sentence vectors

Make classic word vectors more useful for sentence search.

</div>

## When do you need this?

You already have GloVe word vectors and want better sentence comparisons.
You are moving a SIF or uSIF model from fse 1.x to 2.0.
**Prepare the model once**, then use fse to run the finished table.

## The story in pictures

<picture>
<source media="(prefers-color-scheme: dark)" srcset="media/01-sif-usif-dark.svg">
<img src="media/01-sif-usif-light.svg" alt="Training texts teach word weights and common directions. Fold these into the word table, then encode new sentences.">
</picture>

## What changes?

A plain mean gives every word the same weight.
**SIF** (Smooth Inverse Frequency) lowers the weight of common words and removes shared directions from sentence vectors.
**uSIF** estimates the word-weight setting from your corpus and scales the directions it removes.
A direction here is a pattern of numbers shared by many sentences.

## 1. Read sentence pairs

[STS Benchmark](https://ixa2.si.ehu.eus/stswiki/index.php/STSbenchmark) contains sentence pairs with human similarity scores.
We use training texts to prepare the model and keep **test scores out of preparation**.
The shared helper handles temporary folders and downloads, as described in the [index](README.md#run-a-recipe).

In [ ]:
from _helpers import Recipe
import fse
import numpy as np

recipe = Recipe()
work = recipe.work

In [ ]:
from _helpers import ROOT
from _sif import probabilities, word_weights, means, components, remove, unit
from IPython.display import Markdown, display

import numpy as np

In [ ]:
import csv
import re
from scipy.stats import pearsonr
from huggingface_hub import hf_hub_download
from gensim.models import KeyedVectors
from safetensors.numpy import load_file
from tokenizers import Tokenizer

repository = ROOT

def split(name):
    with (repository / "evaluation" / f"sts-{name}.csv").open(encoding="utf-8") as stream:
        rows = list(csv.reader(stream, delimiter="\t", quoting=csv.QUOTE_NONE))
    return [r[5] for r in rows], [r[6] for r in rows], np.array([float(r[4]) for r in rows])
train_a, train_b, _ = split("train")
test_a, test_b, gold = split("test")

source_file = recipe.source_file

glove_repo, glove_revision = "fse/glove-wiki-gigaword-300", "4e5ad402664236b85be45a0d107b95259ba8a909"
# Both files must be beside each other for gensim's native saved format.
glove_files = ("glove-wiki-gigaword-300.model", "glove-wiki-gigaword-300.model.vectors.npy")
glove_paths = [source_file(glove_repo, glove_revision, filename) for filename in glove_files]
if glove_paths[0].parent != glove_paths[1].parent:
    # An incomplete preexisting cache must not split gensim's sidecar pair.
    glove_paths = [Path(hf_hub_download(glove_repo, filename, revision=glove_revision, cache_dir=work / "hub")) for filename in glove_files]
glove = KeyedVectors.load(str(glove_paths[0]), mmap="r")
potion_repo, potion_revision = "minishlab/potion-base-8M", "bf8b056651a2c21b8d2565580b8569da283cab23"
potion_rows = load_file(source_file(potion_repo, potion_revision, "model.safetensors"))["embeddings"]
potion_tokenizer = Tokenizer.from_file(str(source_file(potion_repo, potion_revision, "tokenizer.json")))
potion_words = [potion_tokenizer.id_to_token(i) for i in range(len(potion_rows))]
unknown = potion_tokenizer.token_to_id("[UNK]")

def glove_tokens(text):
    return re.findall(r"[a-z0-9]+(?:'[a-z]+)?", text.lower())
def potion_tokens(text):
    return [potion_words[i] for i in potion_tokenizer.encode(text, add_special_tokens=False).ids if i != unknown]
print({"train_pairs": len(train_a), "test_pairs": len(test_a), "glove_revision": glove_revision, "potion_revision": potion_revision})

## 2. Weight words and remove common patterns

The helper implements the [SIF paper](https://openreview.net/forum?id=SyK00v5xx), [uSIF paper](https://aclanthology.org/W18-3012/), and [fse 1.x choices](https://github.com/oborchers/Fast_Sentence_Embeddings/tree/1.x/src/fse/models).
It finds common directions without first subtracting the average vector, matching those choices.
**Folding** means applying the learned changes to every word row ahead of time.
After folding, averaging those rows gives the changed sentence vector, apart from rounding.

In [ ]:
def score(rows):
    left, right = rows[:len(gold)], rows[len(gold):]
    similarity = np.einsum("ij,ij->i", left, right)
    return float(pearsonr(gold, similarity).statistic * 100)

## 3. Save, reload, and compare

We try a plain mean, SIF with 10 directions, and uSIF with 5 directions on both tables.
GloVe splits lowercase words with a regular expression, while POTION keeps its own word-piece tokenizer.
**Keep that tokenizer with the saved model** because these artifacts expect token lists, rather than raw strings.
The loop also checks folded output against changes applied after averaging.

In [ ]:
results = []
for name, table, words_, tokenize in (
    ("GloVe-300", glove.vectors, glove.index_to_key, glove_tokens),
    ("POTION-8M", potion_rows, potion_words, potion_tokens),
):
    train_tokens = [tokenize(t) for t in train_a + train_b]
    test_tokens = [tokenize(t) for t in test_a + test_b]
    p = probabilities(words_, train_tokens)
    length = max(1, int(np.mean([len(t) for t in train_tokens])))
    for method in ("mean", "sif", "usif"):
        weights = word_weights(p, method, length)
        weighted = np.asarray(table * weights[:, None], dtype=np.float32)
        if method == "mean":
            basis, factors = np.empty((0, table.shape[1]), dtype=np.float32), np.empty(0, dtype=np.float32)
        else:
            fitted = means(weighted, words_, train_tokens)
            basis, factors = components(fitted, 10 if method == "sif" else 5, method == "usif")
        # Fold in bounded blocks, so no vocabulary-sized intermediate projection is needed.
        folded = np.empty_like(weighted)
        for start in range(0, len(weighted), 16384):
            folded[start:start+16384] = remove(weighted[start:start+16384], basis, factors)
        finished = fse.Encoder.from_arrays(folded, words=words_, tokenizer="external", normalize=True)
        # External tokenization is an explicit part of this recipe. Save it alongside the model in a real application.
        path = work / f"{name}-{method}.safetensors"
        finished.save(path)
        finished = fse.Encoder.load(path, mmap=True)
        encoded = finished.encode(test_tokens, workers=1)
        reference = unit(remove(means(weighted, words_, test_tokens), basis, factors))
        error = float(np.max(np.abs(encoded-reference)))
        np.testing.assert_allclose(encoded, reference, atol=3e-5, rtol=3e-5)
        results.append({"table": name, "method": method, "pearson_x100": score(encoded), "fold_max_abs_error": error, "mean_train_length": length})
        del finished, weighted, folded
display(Markdown("| Table | Method | Pearson × 100 |\n|---|---|---:|\n" + "\n".join(
    f"| {r['table']} | {r['method']} | {r['pearson_x100']:.2f} |" for r in results)))
print("Largest folding error:", max(r["fold_max_abs_error"] for r in results))

## Result

Pearson correlation measures how closely the predicted scores follow human scores, and higher is better here.
**Compare methods within each table** because the tokenizers differ between GloVe and POTION.
POTION already contains learned weights, so extra SIF or uSIF is an experiment, not a default recommendation.
These results concern this split only, so evaluate your own texts before changing a deployed model.

## Optional: check the old README numbers

The old experiment prepared its directions using test texts and English word frequencies.
That makes it a **historical diagnostic**, rather than a test of unseen data.
The next cell keeps it separate from the training-only comparison above.

In [ ]:
# Historical diagnostic only: fit on the test texts as the 1.x notebook did.
# Do not use test-fitted components when reporting held-out quality.
import nltk
from nltk import word_tokenize
from wordfreq import get_frequency_dict
nltk.download("punkt_tab", download_dir=str(work / "nltk"), quiet=True, raise_on_error=True)
nltk.data.path.insert(0, str(work / "nltk"))

def historical_tokens(text):
    result = []
    for token in word_tokenize(text):
        if re.match(r".*[A-Za-z0-9].*", token):
            token = token.lower().strip("';.:()").strip('"')
            result.extend(re.split("-", "not" if token == "n't" else token))
    return result
historical = [historical_tokens(t) for t in test_a + test_b]
frequency = get_frequency_dict("en", wordlist="best")
counts = np.array([int(frequency.get(w, 1e-8) * (2**31-1)) for w in glove.index_to_key], dtype=np.int64)
p = counts.astype(np.float32)
p /= counts.sum()
legacy_results = []
for method, published in (("mean", 44.46), ("sif", 71.43), ("usif", 68.82)):
    weighted = glove.vectors * word_weights(p, method, 11)[:, None]
    rows = means(weighted, glove.index_to_key, historical)
    if method != "mean":
        basis, factors = components(rows, 10 if method == "sif" else 5, method == "usif")
        reference = unit(remove(rows, basis, factors))
        folded = np.empty_like(weighted)
        for start in range(0, len(weighted), 16384):
            folded[start:start+16384] = remove(weighted[start:start+16384], basis, factors)
    else:
        reference, folded = unit(rows), weighted
    model = fse.Encoder.from_arrays(folded, words=glove.index_to_key, tokenizer="external", normalize=True)
    # 1.x accepted empty tokens. They contribute no row and only rescale the mean.
    # Drop them at the 2.0 boundary. Cosine direction is unchanged.
    migrated = model.encode([[t for t in tokens if t] for tokens in historical], workers=1)
    np.testing.assert_allclose(migrated, reference, atol=3e-5, rtol=3e-5)
    measured = score(migrated)
    legacy_results.append({"method": method, "recipe_pearson_x100": measured, "README_1_1_0": published, "delta_points": measured-published})
    del model, weighted, folded
for row in legacy_results:
    print(row)

The historical targets are mean 44.46, SIF 71.43, and uSIF 68.82.
The printed differences show whether this run reproduces each target.
**A small numerical difference is still a difference**, so do not claim exact reproduction when the numbers disagree.

## What next?

- [Run a trained linear head](02-linear-heads.ipynb).
- [fse API and design](../docs/architecture/formats.md)
- [GloVe source model and license](https://huggingface.co/fse/glove-wiki-gigaword-300), [POTION model and license](https://huggingface.co/minishlab/potion-base-8M)

Back to the [recipe index](README.md) or the [main README](../README.md#recipes-from-a-model-to-a-working-application).

## Clean up

The helper keeps generated files in a temporary folder. Run this after you finish exploring the results.

In [ ]:
recipe.close()